**check**

In [2]:
import pandas as pd
import numpy as np
import os 
import sys
from pathlib import Path
from tqdm import tqdm
from IPython.display import display

In [3]:
data = r"D:\nam_4\DATN\dataset\Defactify_dataset"
old_metadata = os.path.join(data,"metadata_backup.csv")
new_metadata = os.path.join(data,"metadata.csv")

data_old = pd.read_csv(old_metadata)
data_new = pd.read_csv(new_metadata)

**check rò rỉ dữ liệu**
**in ra các lặp lại trong 1 tập**
**in ra các câu lặp lại giữa các tập**
**xóa (xóa ở đâu, xóa bao nhiêu, vì sao xóa như vậy)**


In [4]:
# 1. Lọc ra đúng 55 caption xuất hiện ở nhiều hơn 1 tập
cap_splits = data_old.groupby('caption')['split'].nunique()
leaked_caps = cap_splits[cap_splits > 1].index

# 2. Tạo bảng
df_leak = (
    data_old[data_old['caption'].isin(leaked_caps)]
    .groupby(['caption', 'split'])['group_id']
    .unique()
    .apply(lambda grps: ', '.join(map(str, sorted(grps))))
    .unstack(fill_value='-')
    [['train', 'validation', 'test']]
    .reset_index()
)

# 3. Đặt lại tên cột và đánh số thứ tự từ 1 đến 55
df_leak.columns = ['Caption', 'Nằm ở Train (Groups)', 'Nằm ở Validation (Groups)', 'Nằm ở Test (Groups)']
df_leak.index = range(1, len(df_leak) + 1)

print(f"=== BẢNG CHI TIẾT {len(df_leak)} CAPTION BỊ RÒ RỈ GIỮA 3 TẬP ===\n")
# Dùng print to_string() để in chữ thuần, không bị lỗi giao diện HTML
print(df_leak.to_string())


=== BẢNG CHI TIẾT 55 CAPTION BỊ RÒ RỈ GIỮA 3 TẬP ===

                                                           Caption          Nằm ở Train (Groups) Nằm ở Validation (Groups)           Nằm ở Test (Groups)
1                            A airplane that is flying in the sky.                          6879                       965                             -
2                    A bathroom sink sitting under a large mirror.                          6859                         -  1342, 2842, 4342, 5842, 7342
3         A bathroom with a white toilet next to a walk in shower.                          2851                         -   370, 1870, 3370, 4870, 6370
4                          A bird perched on top of a tree branch.                          2868                       398   914, 2414, 3914, 5414, 6914
5                             A bus that is sitting in the street.                          2106                      1373                             -
6                           

**Kiểm tra số lượng trước và sau xử lý rò rỉ dữ liệu**

In [19]:
print("=" * 60)
print("[KIỂM TRA 4] THỐNG KÊ SỐ LƯỢNG THEO TỪNG TẬP VÀ MODEL")
print("=" * 60)

# Bảng Pivot số lượng ảnh theo Split x Model
pivot_table = pd.crosstab(
    data_old['split'],
    data_old['model_name'],
    margins=True,
    margins_name="TỔNG CỘNG"
)
display(pivot_table)

# Thống kê tổng quan số group và số ảnh Real/AI
summary = data_old.groupby('split').agg(
    So_Groups=('group_id', 'nunique'),
    Anh_Real=('label_a', lambda x: (x == 0).sum()),
    Anh_AI=('label_a', lambda x: (x == 1).sum()),
    Tong_Anh=('label_a', 'count')
)
display(summary)

# Bảng Pivot số lượng ảnh theo Split x Model
pivot_table1 = pd.crosstab(
    data_new['split'],
    data_new['model_name'],
    margins=True,
    margins_name="TỔNG CỘNG"
)
display(pivot_table1)

# Thống kê tổng quan số group và số ảnh Real/AI
summary2 = data_new.groupby('split').agg(
    So_Groups=('group_id', 'nunique'),
    Anh_Real=('label_a', lambda x: (x == 0).sum()),
    Anh_AI=('label_a', lambda x: (x == 1).sum()),
    Tong_Anh=('label_a', 'count')
)
display(summary2)
print(f"-> Danh sách cột cu: {list(data_old.columns)}")
print(f"-> Danh sách cột moi: {list(data_new.columns)}")

[KIỂM TRA 4] THỐNG KÊ SỐ LƯỢNG THEO TỪNG TẬP VÀ MODEL


model_name,DALL-E_3,MS_COCO,Midjourney_V6,Stable_Diffusion_2.1,Stable_Diffusion_3,Stable_Diffusion_XL,TỔNG CỘNG
split,,,,,,,
test,7500,7500,7500,7500,7500,7500,45000
train,7000,7000,7000,7000,7000,7000,42000
validation,1500,1500,1500,1500,1500,1500,9000
TỔNG CỘNG,16000,16000,16000,16000,16000,16000,96000


,So_Groups,Anh_Real,Anh_AI,Tong_Anh
split,,,,
test,7500,7500,37500,45000
train,7000,7000,35000,42000
validation,1500,1500,7500,9000


model_name,DALL-E_3,MS_COCO,Midjourney_V6,Stable_Diffusion_2.1,Stable_Diffusion_3,Stable_Diffusion_XL,TỔNG CỘNG
split,,,,,,,
test,7330,7330,7330,7330,7330,7330,43980
train,7000,7000,7000,7000,7000,7000,42000
validation,1471,1471,1471,1471,1471,1471,8826
TỔNG CỘNG,15801,15801,15801,15801,15801,15801,94806


,So_Groups,Anh_Real,Anh_AI,Tong_Anh
split,,,,
test,7330,7330,36650,43980
train,7000,7000,35000,42000
validation,1471,1471,7355,8826


-> Danh sách cột cu: ['group_id', 'filepath', 'split', 'label_a', 'label_b', 'model_name', 'caption']
-> Danh sách cột moi: ['group_id', 'filepath', 'split', 'label_a', 'label_b', 'model_name', 'caption', 'group_uid']


**kiểm tra mỗi group có 6 ảnh**

In [18]:
print("=" * 60)
print("[KIỂM TRA 1] CẤU TRÚC GROUP (6 ẢNH = 1 REAL + 5 AI)")
print("=" * 60)

# 1. Đếm tổng số ảnh theo cặp (split, group_id)
group_sizes = data_new.groupby(['split', 'group_id']).size()
bad_size = group_sizes[group_sizes != 6]

if len(bad_size) == 0:
    print(f"  [PASS] TUYỆT VỜI! 100% các groups ({len(group_sizes):,} groups) đều có đúng 6 ảnh.")
else:
    print(f"  [FAIL] Phát hiện {len(bad_size)} groups không đủ 6 ảnh: {bad_size.head()}")

# 2. Đếm số ảnh Real và AI trong mỗi group
group_labels = data_new.groupby(['split', 'group_id'])['label_a'].agg(
    real_count=lambda x: (x == 0).sum(),
    ai_count=lambda x: (x == 1).sum()
)

invalid_groups = group_labels[(group_labels['real_count'] != 1) | (group_labels['ai_count'] != 5)]

if len(invalid_groups) == 0:
    print(f"  [PASS] TUYỆT VỜI! 100% các groups đều có đúng chuẩn 1 ảnh Real và 5 ảnh AI.")
else:
    print(f"  [FAIL] Có {len(invalid_groups)} groups sai tỷ lệ Real/AI:\n", invalid_groups.head())


[KIỂM TRA 1] CẤU TRÚC GROUP (6 ẢNH = 1 REAL + 5 AI)
  [PASS] TUYỆT VỜI! 100% các groups (15,801 groups) đều có đúng 6 ảnh.
  [PASS] TUYỆT VỜI! 100% các groups đều có đúng chuẩn 1 ảnh Real và 5 ảnh AI.


In [13]:
print("=" * 60)
print("[KIỂM TRA 2] ĐỘ KHỚP NHAU GIỮA LABEL_A, LABEL_B VÀ MODEL_NAME")
print("=" * 60)

LABEL_B_MAP = {
    0: "MS_COCO",
    1: "Stable_Diffusion_2.1",
    2: "Stable_Diffusion_XL",
    3: "Stable_Diffusion_3",
    4: "DALL-E_3",
    5: "Midjourney_V6",
}

# 1. Kiểm tra label_a và label_b
bad_label_a_0 = data_new[(data_new['label_a'] == 0) & (data_new['label_b'] != 0)]
bad_label_a_1 = data_new[(data_new['label_a'] == 1) & (data_new['label_b'] == 0)]

# 2. Kiểm tra label_b và model_name
expected_models = data_new['label_b'].map(LABEL_B_MAP)
mismatched_models = data_new[data_new['model_name'] != expected_models]

if len(bad_label_a_0) == 0 and len(bad_label_a_1) == 0 and len(mismatched_models) == 0:
    print("  [PASS] 100% các dòng đều khớp nhãn hoàn hảo:")
    print("         - label_a == 0 đi cùng label_b == 0 (MS_COCO)")
    print("         - label_a == 1 đi cùng label_b trong [1..5] và model_name chính xác.")
else:
    print(f"  [FAIL] Phát hiện sai lệch:")
    print(f"    - Real nhưng label_b != 0 : {len(bad_label_a_0)}")
    print(f"    - AI nhưng label_b == 0   : {len(bad_label_a_1)}")
    print(f"    - Sai lệch tên model_name : {len(mismatched_models)}")


[KIỂM TRA 2] ĐỘ KHỚP NHAU GIỮA LABEL_A, LABEL_B VÀ MODEL_NAME
  [PASS] 100% các dòng đều khớp nhãn hoàn hảo:
         - label_a == 0 đi cùng label_b == 0 (MS_COCO)
         - label_a == 1 đi cùng label_b trong [1..5] và model_name chính xác.


In [ ]:
print("=" * 60)
print("[KIỂM TRA 3] ĐƯỜNG DẪN VÀ FILE THỰC TẾ TRÊN Ổ CỨNG")
print("=" * 60)

missing_files = []
zero_byte_files = []

# Quét kiểm tra tất cả các file trong metadata
for _, row in tqdm(data_new.iterrows(), total=len(data_new), desc="Đang kiểm tra file"):
    abs_path = os.path.join(data, row['filepath'])
    if not os.path.exists(abs_path):
        missing_files.append(row['filepath'])
    elif os.path.getsize(abs_path) == 0:
        zero_byte_files.append(row['filepath'])

if len(missing_files) == 0 and len(zero_byte_files) == 0:
    print(f"\n  [PASS] Tất cả {len(data_new):,} file trong metadata đều tồn tại và hợp lệ trên ổ đĩa!")
else:
    print(f"\n  [FAIL] File bị thiếu : {len(missing_files)}")
    print(f"  [FAIL] File rỗng 0KB  : {len(zero_byte_files)}")


[KIỂM TRA 3] ĐƯỜNG DẪN VÀ FILE THỰC TẾ TRÊN Ổ CỨNG


Đang kiểm tra file: 100%|██████████| 94806/94806 [00:13<00:00, 7179.41it/s]


  [PASS] Tất cả 94,806 file trong metadata đều tồn tại và hợp lệ trên ổ đĩa!


In [21]:
print("=" * 70)
print("[KIỂM TRA 5] KIỂM TRA TÍNH TOÀN VẸN CỦA NHÓM (GROUP INTEGRITY)")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. KIỂM TRA CẤU TRÚC: Mỗi nhóm (split, group_id) phải có đúng 6 ảnh (1 Real + 5 AI)
# ----------------------------------------------------------------------
group_counts = data_new.groupby(['split', 'group_id']).agg(
    total_imgs=('label_a', 'count'),
    real_imgs=('label_a', lambda x: (x == 0).sum()),
    ai_imgs=('label_a', lambda x: (x == 1).sum()),
    unique_captions=('caption', 'nunique')
)

bad_structure = group_counts[
    (group_counts['total_imgs'] != 6) | 
    (group_counts['real_imgs'] != 1) | 
    (group_counts['ai_imgs'] != 5) |
    (group_counts['unique_captions'] != 1)
]

# ----------------------------------------------------------------------
# 2. KIỂM TRA PHÂN TẬP: Cả 6 ảnh của cùng 1 nhóm có bị xẻ đôi sang tập khác không?
#    (Một caption đại diện cho 1 bối cảnh nhóm - chỉ được phép nằm ở đúng 1 split)
# ----------------------------------------------------------------------
caption_splits = data_new.groupby('caption')['split'].nunique()
leaked_groups = caption_splits[caption_splits > 1]

# ----------------------------------------------------------------------
# 3. KẾT LUẬN VÀ BÁO CÁO
# ----------------------------------------------------------------------
if len(bad_structure) == 0 and len(leaked_groups) == 0:
    print(f"  [PASS] HOÀN TOÀN CHÍNH XÁC VÀ TOÀN VẸN 100%!")
    print(f"         • Tổng số nhóm thực tế: {len(group_counts):,} groups.")
    print(f"         • Cấu trúc mỗi nhóm: ĐÚNG 6 ảnh (1 Real + 5 AI, cùng 1 caption).")
    print(f"         • Tính toàn vẹn: 100% các nhóm nằm trọn vẹn trong DUY NHẤT một tập.")
    print(f"         • Độ phân tán/rò rỉ giữa các tập: 0 nhóm (Không có ảnh nào bị chia tách).")
    print(f"         => KẾT LUẬN: Đạt chuẩn nguyên tắc theo nhóm. Dữ liệu sẵn sàng để huấn luyện!")
else:
    print("  [FAIL] Phát hiện bất thường:")
    if len(bad_structure) > 0:
        print(f"    - Có {len(bad_structure)} nhóm sai cấu trúc 6 ảnh!")
    if len(leaked_groups) > 0:
        print(f"    - Có {len(leaked_groups)} nhóm bị phân tán giữa các tập!")


[KIỂM TRA 5] KIỂM TRA TÍNH TOÀN VẸN CỦA NHÓM (GROUP INTEGRITY)
  [PASS] HOÀN TOÀN CHÍNH XÁC VÀ TOÀN VẸN 100%!
         • Tổng số nhóm thực tế: 15,801 groups.
         • Cấu trúc mỗi nhóm: ĐÚNG 6 ảnh (1 Real + 5 AI, cùng 1 caption).
         • Tính toàn vẹn: 100% các nhóm nằm trọn vẹn trong DUY NHẤT một tập.
         • Độ phân tán/rò rỉ giữa các tập: 0 nhóm (Không có ảnh nào bị chia tách).
         => KẾT LUẬN: Đạt chuẩn nguyên tắc theo nhóm. Dữ liệu sẵn sàng để huấn luyện!


In [16]:
# Kiểm tra độ giao thoa trên file metadata.csv hiện tại (data_new)
tr = set(data_new[data_new['split'] == 'train']['caption'].dropna().unique())
va = set(data_new[data_new['split'] == 'validation']['caption'].dropna().unique())
te = set(data_new[data_new['split'] == 'test']['caption'].dropna().unique())

print("=== KIỂM CHỨNG DATA LEAKAGE TRÊN METADATA HIỆN TẠI ===")
print(f"• Trùng giữa Train và Test      : {len(tr & te)} caption")
print(f"• Trùng giữa Train và Validation: {len(tr & va)} caption")
print(f"• Trùng giữa Val và Test        : {len(va & te)} caption")
print(f"• Xuất hiện ở cả 3 tập          : {len(tr & va & te)} caption")



=== KIỂM CHỨNG DATA LEAKAGE TRÊN METADATA HIỆN TẠI ===
• Trùng giữa Train và Test      : 0 caption
• Trùng giữa Train và Validation: 0 caption
• Trùng giữa Val và Test        : 0 caption
• Xuất hiện ở cả 3 tập          : 0 caption
